In [1]:
!pip install numpy pandas matplotlib torch==2.9.0 torchaudio==2.9.0 torchcodec==0.8.0 datasets==4.4.1 transformers==4.57.3 evaluate==0.4.6 accelerate==1.12 jiwer tensorboard

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os
import torch
import numpy as np
import re
import gc
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_from_disk, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model, TaskType

DATASET_PATH = "/content/drive/MyDrive/Processed_Hindi_Parquet_Dataset"
OUTPUT_DIR   = "/content/drive/MyDrive/whisper-small-lora-ap"

MODEL_ID = "openai/whisper-small"
LANGUAGE = "hindi"
TASK = "transcribe"
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 1
NUM_TRAIN_EPOCHS = 3
LEARNING_RATE = 1e-5
WARMUP_STEPS = 50
EVAL_STEPS = 200
SAVE_STEPS = 200


if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA GPU")
else:
    device = torch.device("cpu")
    print("Using CPU (⚠️ Slow)")


metric_wer = evaluate.load("wer")
metric_cer = evaluate.load("cer")


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100
        )

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# --- PREPROCESSING ---
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"],
        sampling_rate=audio["sampling_rate"]
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

# --- METRICS COMPUTATION ---
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {
        "wer": 100 * metric_wer.compute(predictions=pred_str, references=label_str),
        "cer": 100 * metric_cer.compute(predictions=pred_str, references=label_str),
    }

# --- MAIN ---
if __name__ == "__main__":

    print("Loading processor and model...")
    processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)

    model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
    model.to(device)

    # --- FIX DECODER CONFIGURATION ---
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []

    model.generation_config.language = LANGUAGE
    model.generation_config.task = TASK
    model.generation_config.forced_decoder_ids = None

    # --- APPLY LORA ---
    print("Applying LoRA...")
    peft_config = LoraConfig(
        inference_mode=False,
        r=4,
        lora_alpha=4,
        target_modules=["q_proj", "k_proj"],
        lora_dropout=0.1,
        bias="none",
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    # --- LOAD DATASET ---
    print("Loading dataset...")
    ds = load_from_disk(DATASET_PATH)

    if "validation" not in ds:
        ds_splits = ds.train_test_split(test_size=50, seed=42)
        ds = DatasetDict({"train": ds_splits["train"], "validation": ds_splits["test"]})

    if os.path.exists("/content/drive/MyDrive/Processed_Hindi_Mapped"):
        ds_prepared = load_from_disk("/content/drive/MyDrive/Processed_Hindi_Mapped")
    else:
        ds_prepared = ds.map(
            prepare_dataset,
            remove_columns=ds["train"].column_names,
            num_proc=1,
        )
        ds_prepared.save_to_disk("/content/drive/MyDrive/Processed_Hindi_Mapped")

    data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

    # --- TRAINING ARGS ---
    training_args = Seq2SeqTrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_TRAIN_EPOCHS,

        eval_strategy="steps",
        eval_steps=EVAL_STEPS,

        fp16=True,
        bf16=False,

        predict_with_generate=True,
        generation_max_length=225,

        save_steps=SAVE_STEPS,
        save_total_limit=2,

        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,

        remove_unused_columns=False,
        label_names=["labels"],
        max_grad_norm=1.0,
        report_to = "none"
    )

    # --- TRAINER ---
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=ds_prepared["train"],
        eval_dataset=ds_prepared["validation"],
        data_collator=data_collator,
        tokenizer=processor.feature_extractor,
        compute_metrics=compute_metrics,
    )

    # --- TRAIN ---
    print("Starting training...")
    trainer.train()

    # --- SAVE ---
    model.save_pretrained(OUTPUT_DIR)
    processor.save_pretrained(OUTPUT_DIR)

    print("Training Done!")

Using CUDA GPU
Loading processor and model...
Applying LoRA...
trainable params: 442,368 || all params: 242,177,280 || trainable%: 0.1827
Loading dataset...


Map (num_proc=1):   0%|          | 0/5517 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/50 [00:00<?, ? examples/s]

Saving the dataset (0/11 shards):   0%|          | 0/5517 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/50 [00:00<?, ? examples/s]

...

You're using a WhisperTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss,Validation Loss,Wer,Cer
200,2.165100,2.264113,100.743746,74.095639
400,2.015800,2.101268,105.273834,74.058474
600,1.884100,2.002671,104.124408,73.253221
800,1.825300,1.932352,105.409060,75.532706
1000,1.795000,1.886879,98.580122,70.490585
1200,1.773000,1.854765,96.416498,69.722498


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Step,Training Loss,Validation Loss,Wer,Cer
200,2.165100,2.264113,100.743746,74.095639
400,2.015800,2.101268,105.273834,74.058474
600,1.884100,2.002671,104.124408,73.253221
800,1.825300,1.932352,105.409060,75.532706
1000,1.795000,1.886879,98.580122,70.490585
1200,1.773000,1.854765,96.416498,69.722498
1400,1.761200,1.830673,94.590940,70.044599
1600,1.723900,1.813721,91.751183,68.012884
1800,1.714700,1.803008,91.548343,67.653617
2000,1.711200,1.798178,94.590940,69.226957


Training Done!
